# 01 — Khám phá dữ liệu (EDA)

Phụ trách: **Thành viên 1**. EDA trên tập phát hiện biển báo (thống kê box, không crop từng biển).

## 0) Bootstrap (local / Kaggle / Colab)

- **Local**: notebook nằm trong repo nên dùng luôn repo hiện tại (không đụng vào git của bạn).
- **Kaggle / Colab**: clone repo về `/kaggle/working`, lấy đúng nhánh rồi cài thư viện. Trên Kaggle bật **Internet: On**.
- Repo private: xem `README.md` mục 12.8 (token lưu trong Kaggle Secrets).

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "TODO_URL"             # TODO: URL repo GitHub của nhóm
REPO_DIR = "TSD-Voice-Alert"
REPO_BRANCH = "main"              # Thành viên 1: data-yolo · Thành viên 2: faster-rcnn (khi nhánh đã có trên origin)

ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB = "google.colab" in sys.modules

def sh(cmd: str, check: bool = True) -> int:
    print(">>", cmd)
    return subprocess.run(cmd, shell=True, check=check).returncode

# 1. Tìm repo đang chứa notebook này (local)
local_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "ts_alert").is_dir() and (p / "configs" / "shared.yaml").exists()), None)

if local_root is not None:
    repo_path = local_root
    print("Dùng repo hiện tại:", repo_path)
else:
    # 2. Chưa có repo: clone rồi lấy đúng nhánh (chỉ làm trong môi trường cloud)
    assert REPO_URL != "TODO_URL", "Điền REPO_URL ở ô này trước khi chạy trên Kaggle/Colab."
    base = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
    repo_path = base / REPO_DIR
    if not (repo_path / ".git").exists():
        sh(f'git clone --branch {REPO_BRANCH} {REPO_URL} "{repo_path}"')
    git = f'git -C "{repo_path}"'
    sh(f"{git} fetch origin {REPO_BRANCH}")
    sh(f"{git} checkout -B {REPO_BRANCH} origin/{REPO_BRANCH}")
    sh(f"{git} log --oneline -1")

os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
print("cwd:", Path.cwd())

# Cài thư viện: chỉ trên Kaggle/Colab. Local thì dùng môi trường đã cài bằng requirements.txt.
if ON_KAGGLE or ON_COLAB:
    sh('pip install -q -e .')

import ts_alert
print("Bootstrap OK.")

In [ ]:
RUNTIME = "configs/runtime/kaggle.yaml" if ON_KAGGLE else "configs/runtime/local.yaml"
print("Runtime:", RUNTIME)

## 1) Cấu hình và đường dẫn dữ liệu

Trên Kaggle: Add Input → `maitam/vietnamese-traffic-signs`, kiểm tra tên thư mục trong `configs/runtime/kaggle.yaml` (`paths.raw_dir`).

In [ ]:
from ts_alert.config.resolve import resolve_config, resolve_path

cfg = resolve_config(None, RUNTIME)
DATA_ROOT = resolve_path(cfg["paths"]["raw_dir"])
print("DATA_ROOT =", DATA_ROOT, "| tồn tại:", DATA_ROOT.exists())

## 2) Chạy các hàm EDA

Các hàm trong `analysis/eda.py` ghi hình vào `analysis/figures/`. Tương đương dòng lệnh: `python -m analysis.eda --data-root <DATA_PATH>`.

In [ ]:
from analysis import eda

eda.FIG_DIR.mkdir(parents=True, exist_ok=True)
eda.class_distribution(DATA_ROOT)     # số box theo từng lớp
eda.box_size_stats(DATA_ROOT)         # kích thước box (tương đối theo ảnh)
eda.center_heatmap(DATA_ROOT)         # vị trí tâm box -> cơ sở cho quy tắc trái/phải/trước mặt

## 3) Xem các hình đã sinh

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

figs = sorted(eda.FIG_DIR.glob("*.png"))
for f in figs:
    plt.figure(figsize=(8, 5))
    plt.imshow(mpimg.imread(f))
    plt.title(f.name)
    plt.axis("off")
plt.show()
print(f"{len(figs)} hình")

## 4) Nhận xét

TODO: Thành viên 1 điền.

- Dataset có bao nhiêu lớp, ảnh, box? (điền vào `configs/classes.csv`, `data.num_classes`, README mục 6)
- Phân bố lớp có mất cân bằng không? Lớp nào ít mẫu nhất?
- Biển nhỏ đến mức nào (tỉ lệ so với ảnh)? Ảnh hưởng gì đến YOLO và Faster R-CNN?
- Heatmap tâm box gợi ý ngưỡng trái/phải/trước mặt (`postprocess.position` trong `shared.yaml`) là bao nhiêu?